# Grid-wise MFC Decomposition over `TARGET_BOX`

This notebook estimates ENSO-related lower-tropospheric moisture flux convergence (MFC) decomposition on the Maritime Continent grid and compares:
- `P1 = 1981-2006`
- `P2 = 2007-2025`

Computation domain:
- `mc_extent = [80, 160, -20, 20]`

Reference box:
- lon `95E-125E`
- lat `6S-2N`

Inputs:
- ERA5 pressure-level `q`, `u`, `v`
- DJF seasonal means, where `DJF 1981 = Dec 1980 + Jan/Feb 1981`
- Nino3.4 DJF index standardized over the 1991-2020 climatology

The decomposition follows the period-specific climatology in the equations:
- `qbar`, `ubar`, `vbar` are computed separately for each period before anomalies are formed.

Outputs:
- `moisture_decomp_gridwise_TARGETBOX.nc`
- `moisture_decomp_gridwise_TARGETBOX.png`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.patches import Rectangle

REPO_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing').resolve()
DATA_DIR = REPO_ROOT / 'external' / 'ClimateData'
ERA5_DIR = DATA_DIR / 'era5-monthly'
INDEX_DIR = DATA_DIR / 'index-monthly'
NOTEBOOK_DIR = REPO_ROOT / 'notebooks' / 'dekomposisi_mfc'
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

Q_PATH = ERA5_DIR / 'specific_humidity_1980-2025_1000hpa-100hpa.nc'
UV_PATH = ERA5_DIR / 'uv_wind_1980-2025_1000hpa-100hpa.nc'
NINO34_PATH = INDEX_DIR / 'nino34.anom.csv'
NINO34_COLUMN = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

OUT_NC = NOTEBOOK_DIR / 'moisture_decomp_gridwise_TARGETBOX.nc'
OUT_PNG = NOTEBOOK_DIR / 'moisture_decomp_gridwise_TARGETBOX.png'

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}

MC_EXTENT = [80.0, 160.0, -20.0, 20.0]
MC_XTICKS = np.arange(90, 161, 20)
MC_YTICKS = np.arange(-20, 21, 10)

START = '1980-12-01'
END = '2025-02-28'
FULL_YEARS = np.arange(1981, 2026)
P1_YEARS = np.arange(1981, 2007)
P2_YEARS = np.arange(2007, 2026)
DJF_MONTHS = (12, 1, 2)

G = 9.80665
R_EARTH = 6371000.0

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 200,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## Helpers and physical meaning

- `total`: regression of the full moisture flux `q*u` or `q*v`
- `dynamic`: circulation change acting on mean moisture
- `thermodynamic`: moisture change acting on mean circulation
- `nonlinear`: interaction between moisture and wind anomalies
- `residual`: closure check, `total - dynamic - thermodynamic - nonlinear`
- Positive MFC means moisture convergence


In [ ]:
def standardize_lat_lon(ds):
    rename_map = {}
    if 'latitude' in ds.coords or 'latitude' in ds.dims:
        rename_map['latitude'] = 'lat'
    if 'longitude' in ds.coords or 'longitude' in ds.dims:
        rename_map['longitude'] = 'lon'
    if rename_map:
        ds = ds.rename(rename_map)
    return ds


def djf_seasonal_mean(da):
    # Compute DJF means and assign December to the following DJF year.
    da = da.sel(time=slice(START, END))
    month_mask = da.time.dt.month.isin(DJF_MONTHS)
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.sel(time=month_mask).assign_coords(
        djf_year=('time', djf_year.sel(time=month_mask).data)
    )
    da = da.where(da.djf_year.isin(FULL_YEARS), drop=True)
    return da.groupby('djf_year').mean('time')


def select_djf_years(da, years):
    # Use a boolean mask so the notebook does not depend on djf_year being an indexed coordinate.
    return da.where(da.djf_year.isin(years), drop=True)


def regress_1d(y, x, dim='djf_year'):
    # Ordinary least-squares slope of y onto x along the DJF-year dimension.
    y, x = xr.align(y, x, join='inner')
    x_anom = x - x.mean(dim)
    y_anom = y - y.mean(dim)
    return (y_anom * x_anom).mean(dim) / x_anom.var(dim)


def vertical_integrate(term):
    # Integrate a level-dependent term from 1000 to 700 hPa in Pa.
    term = term.sortby('level').chunk({'level': -1})
    term = term.assign_coords(level=('level', term['level'].values * 100.0))
    return term.integrate('level') / G


def mfc_from_flux(q_u, q_v):
    # Compute MFC on the sphere from vertically integrated flux components.
    q_u, q_v = xr.align(q_u, q_v, join='inner')
    q_u = q_u.sortby('lat')
    q_v = q_v.sortby('lat')

    lat_rad = np.deg2rad(q_u['lat'])
    dqu_dlon = q_u.differentiate('lon') * (180.0 / np.pi)
    dqv_dlat = q_v.differentiate('lat') * (180.0 / np.pi)

    # Positive MFC means convergence.
    return (-(dqu_dlon / (R_EARTH * np.cos(lat_rad)) + dqv_dlat / R_EARTH)).rename('mfc')


def add_target_box(ax):
    ax.add_patch(
        Rectangle(
            (TARGET_BOX['lon_min'], TARGET_BOX['lat_min']),
            TARGET_BOX['lon_max'] - TARGET_BOX['lon_min'],
            TARGET_BOX['lat_max'] - TARGET_BOX['lat_min'],
            fill=False,
            edgecolor='black',
            linewidth=1.5,
            linestyle='--',
            transform=ccrs.PlateCarree(),
            zorder=5,
        )
    )

def _sanitize_attr_value(value):
    if isinstance(value, (str, bytes, int, float, np.integer, np.floating, np.bool_)) or value is None:
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple, np.ndarray, pd.Index)):
        return [str(v) if pd.isna(v) else v for v in list(value)]
    return str(value)


def _coerce_string_like_array(da):
    values = np.asarray(da.values)
    flat = values.reshape(-1)
    coerced = ['' if pd.isna(v) else str(v) for v in flat]
    arr = np.asarray(coerced, dtype=str).reshape(values.shape)
    return xr.DataArray(arr, coords=da.coords, dims=da.dims, attrs=da.attrs, name=da.name)


def sanitize_for_netcdf(ds):
    ds = ds.copy()
    ds = ds.reset_coords(drop=True)

    for name in list(ds.data_vars):
        dtype = ds[name].dtype
        if pd.api.types.is_float_dtype(dtype):
            ds[name] = ds[name].astype('float32')
        elif pd.api.types.is_bool_dtype(dtype):
            ds[name] = ds[name].astype('int8')
        elif pd.api.types.is_string_dtype(dtype):
            ds[name] = _coerce_string_like_array(ds[name])
        elif dtype == object:
            values = np.asarray(ds[name].values, dtype=object).reshape(-1)
            if all(pd.isna(v) or isinstance(v, (str, bytes)) for v in values):
                ds[name] = _coerce_string_like_array(ds[name])
        ds[name].attrs = {k: _sanitize_attr_value(v) for k, v in ds[name].attrs.items()}

    for name in list(ds.coords):
        dtype = ds[name].dtype
        if pd.api.types.is_string_dtype(dtype):
            ds = ds.assign_coords({name: _coerce_string_like_array(ds[name])})
        elif dtype == object:
            values = np.asarray(ds[name].values, dtype=object).reshape(-1)
            if all(pd.isna(v) or isinstance(v, (str, bytes)) for v in values):
                ds = ds.assign_coords({name: _coerce_string_like_array(ds[name])})
        ds[name].attrs = {k: _sanitize_attr_value(v) for k, v in ds[name].attrs.items()}

    ds.attrs = {k: _sanitize_attr_value(v) for k, v in ds.attrs.items()}
    return ds


def write_dataset(ds, path):
    ds = sanitize_for_netcdf(ds)
    tmp_path = path.with_name(f'.{path.name}.tmp')
    tmp_path.unlink(missing_ok=True)
    ds.to_netcdf(tmp_path)
    tmp_path.replace(path)



## Load ERA5 and Nino3.4

The datasets are opened lazily, then subset to the Maritime Continent plotting domain and the 1000-700 hPa layer.


In [ ]:
# Open q, u, and v lazily, then subset early to the MC domain and lower troposphere.
# Keep coordinate renaming conditional because some ERA5 files already use lat/lon.
q_ds = xr.open_dataset(Q_PATH, drop_variables=['step'], chunks={'time': 12})[['q']]
q_ds = standardize_lat_lon(q_ds)
q_ds = q_ds.assign_coords(lon=(q_ds.lon % 360)).sortby('lon')

uv_ds = xr.open_dataset(UV_PATH, drop_variables=['step'], chunks={'time': 12})[['u', 'v']]
uv_ds = standardize_lat_lon(uv_ds)
uv_ds = uv_ds.assign_coords(lon=(uv_ds.lon % 360)).sortby('lon')

lat_values = np.asarray(q_ds['lat'].values)
lat_slice = (
    slice(MC_EXTENT[2], MC_EXTENT[3])
    if lat_values[0] < lat_values[-1]
    else slice(MC_EXTENT[3], MC_EXTENT[2])
)
selected_levels = [float(lev) for lev in np.asarray(q_ds['level'].values) if 700 <= lev <= 1000]

target_lon_slice = slice(MC_EXTENT[0], MC_EXTENT[1])

q_ds = q_ds.sel(
    time=slice(START, END),
    level=selected_levels,
    lat=lat_slice,
    lon=target_lon_slice,
).sortby('lat')

uv_ds = uv_ds.sel(
    time=slice(START, END),
    level=selected_levels,
    lat=lat_slice,
    lon=target_lon_slice,
).sortby('lat')

q = q_ds['q']
u = uv_ds['u']
v = uv_ds['v']

print('Selected levels (hPa):', selected_levels)
print('q dims:', q.dims, 'shape:', q.shape)
print('u dims:', u.dims, 'shape:', u.shape)
print('v dims:', v.dims, 'shape:', v.shape)
print('Grid extent:', MC_EXTENT)
print('Reference target box:', TARGET_BOX)


## DJF seasonal means and Nino3.4 standardization

Nino3.4 is standardized using the 1991-2020 DJF climatology so the P1 and P2 regression coefficients share the same scale.


In [ ]:
q_djf = djf_seasonal_mean(q)
u_djf = djf_seasonal_mean(u)
v_djf = djf_seasonal_mean(v)

nino_df = pd.read_csv(
    NINO34_PATH,
    usecols=['Date', NINO34_COLUMN],
    parse_dates=['Date'],
)
nino_df = nino_df.set_index('Date').loc[START:END].copy()
nino_df[NINO34_COLUMN] = pd.to_numeric(nino_df[NINO34_COLUMN], errors='coerce')
nino_df[NINO34_COLUMN] = nino_df[NINO34_COLUMN].replace(-9999.0, np.nan)
nino_df = nino_df[nino_df.index.month.isin(DJF_MONTHS)].copy()
nino_df['djf_year'] = nino_df.index.year + (nino_df.index.month == 12).astype('int8')

nino_djf = nino_df.groupby('djf_year')[NINO34_COLUMN].mean().loc[FULL_YEARS]
nino_djf = xr.DataArray(
    nino_djf.to_numpy(),
    coords={'djf_year': nino_djf.index.to_numpy()},
    dims='djf_year',
    name='nino34',
)
nino_clim = select_djf_years(nino_djf, np.arange(1991, 2021))
nino_djf_std = (nino_djf - nino_clim.mean('djf_year')) / nino_clim.std('djf_year', ddof=0)

print('DJF years in q/u/v:', int(q_djf.sizes['djf_year']))
print('DJF years in Nino3.4:', int(nino_djf_std.sizes['djf_year']))
print('Nino3.4 climatology years:', int(nino_clim.sizes['djf_year']))
print('Standardized Nino3.4 mean (full series):', float(nino_djf_std.mean('djf_year')))
print('Standardized Nino3.4 std (1991-2020 baseline):', float(nino_clim.std('djf_year', ddof=0)))
print('q DJF year range:', int(q_djf.djf_year.values[0]), 'to', int(q_djf.djf_year.values[-1]))


## Period-wise decomposition

For each period, compute the period climatology first, then regress the flux terms onto the standardized Nino3.4 index on the native grid.


In [ ]:
period_years = {
    'P1': P1_YEARS,
    'P2': P2_YEARS,
}

term_order = ['total', 'dynamic', 'thermodynamic', 'nonlinear', 'residual']
period_terms = {}

for period_name, years in period_years.items():
    q_p = select_djf_years(q_djf, years)
    u_p = select_djf_years(u_djf, years)
    v_p = select_djf_years(v_djf, years)
    n_p = select_djf_years(nino_djf_std, years)

    q_p, u_p, v_p, n_p = xr.align(q_p, u_p, v_p, n_p, join='inner')

    qbar = q_p.mean('djf_year')
    ubar = u_p.mean('djf_year')
    vbar = v_p.mean('djf_year')

    q_anom = q_p - qbar
    u_anom = u_p - ubar
    v_anom = v_p - vbar

    # Total response: regression of the full moisture flux q*u or q*v.
    total_u = regress_1d(q_p * u_p, n_p)
    total_v = regress_1d(q_p * v_p, n_p)

    # Dynamic response: circulation change acting on the mean moisture field.
    dynamic_u = qbar * regress_1d(u_anom, n_p)
    dynamic_v = qbar * regress_1d(v_anom, n_p)

    # Thermodynamic response: moisture change acting on the mean circulation.
    thermodynamic_u = ubar * regress_1d(q_anom, n_p)
    thermodynamic_v = vbar * regress_1d(q_anom, n_p)

    # Nonlinear response: interaction between moisture and wind anomalies.
    nonlinear_u = regress_1d(q_anom * u_anom, n_p)
    nonlinear_v = regress_1d(q_anom * v_anom, n_p)

    flux_components = {
        'total': (total_u, total_v),
        'dynamic': (dynamic_u, dynamic_v),
        'thermodynamic': (thermodynamic_u, thermodynamic_v),
        'nonlinear': (nonlinear_u, nonlinear_v),
    }

    component_mfc = {}
    for component_name, (flux_u, flux_v) in flux_components.items():
        q_u = vertical_integrate(flux_u)
        q_v = vertical_integrate(flux_v)
        component_mfc[component_name] = mfc_from_flux(q_u, q_v)

    residual_mfc = (
        component_mfc['total']
        - component_mfc['dynamic']
        - component_mfc['thermodynamic']
        - component_mfc['nonlinear']
    )
    component_mfc['residual'] = residual_mfc

    period_terms[period_name] = xr.concat(
        [component_mfc[term] for term in term_order],
        dim=xr.IndexVariable('term', term_order),
    )

    residual_max_abs = float(np.abs(residual_mfc).max().compute())
    residual_mean_abs = float(np.abs(residual_mfc).mean().compute())
    print(f'{period_name} residual max abs: {residual_max_abs:.4e} kg m^-2 s^-1')
    print(f'{period_name} residual mean abs: {residual_mean_abs:.4e} kg m^-2 s^-1')


## Export grid-wise fields

The output NetCDF stores the period-wise decomposition on the native grid, plus the `P2 - P1` difference. Values stay in `kg m^-2 s^-1`.


In [ ]:
period_term = xr.concat(
    [period_terms['P1'].expand_dims(period=['P1']), period_terms['P2'].expand_dims(period=['P2'])],
    dim='period',
).assign_coords(period=np.asarray(['P1', 'P2'], dtype=object))
period_term.name = 'mfc_decomp'

# Make the term coordinate a plain object array to avoid pandas StringDtype during NetCDF export.
period_term = period_term.assign_coords(term=xr.IndexVariable('term', np.asarray(term_order, dtype=object)))

delta_term = (period_term.sel(period='P2') - period_term.sel(period='P1')).rename('mfc_decomp_delta')

out_ds = xr.Dataset(
    {
        'period_term': period_term,
        'delta_term': delta_term,
    }
)

write_dataset(out_ds, OUT_NC)
print(f'Saved NetCDF -> {OUT_NC}')
print(out_ds)


grid_ds = xr.open_dataset(OUT_NC)
period_term = grid_ds['period_term']
delta_term = grid_ds['delta_term']

print(f'Loaded NetCDF -> {OUT_NC}')
print(period_term)
print(delta_term)


In [ ]:
for period_name in ['P1', 'P2']:
    residual = period_term.sel(period=period_name, term='residual')
    total = period_term.sel(period=period_name, term='total')
    print(f'{period_name} total range: {float(total.min().compute()):+.4e} to {float(total.max().compute()):+.4e} kg m^-2 s^-1')
    print(f'{period_name} residual range: {float(residual.min().compute()):+.4e} to {float(residual.max().compute()):+.4e} kg m^-2 s^-1')
    print(f'{period_name} residual max abs: {float(np.abs(residual).max().compute()):.4e} kg m^-2 s^-1')

for term in term_order:
    delta = delta_term.sel(term=term)
    print(f'Delta {term} max abs: {float(np.abs(delta).max().compute()):.4e} kg m^-2 s^-1')


## Map panels

Each term is shown with `P1`, `P2`, and `P2 - P1` panels, with the target box drawn for reference.


In [ ]:
panel_defs = [
    ('P1', 'P1 (1981-2006)', 'period_term', 'P1'),
    ('P2', 'P2 (2007-2025)', 'period_term', 'P2'),
    ('P2-P1', 'P2 - P1', 'delta_term', None),
]
term_labels = {
    'total': 'Total',
    'dynamic': 'Dynamic',
    'thermodynamic': 'Thermodynamic',
    'nonlinear': 'Nonlinear',
    'residual': 'Residual',
}

map_cmap = 'RdBu_r'

for term in term_order:
    fig = plt.figure(figsize=(16, 5.8))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 0.06], hspace=0.18, wspace=0.08)
    axes = [fig.add_subplot(gs[0, i], projection=ccrs.PlateCarree(central_longitude=180)) for i in range(3)]
    cax = fig.add_subplot(gs[1, :])

    row_arrays = [
        period_term.sel(period='P1', term=term),
        period_term.sel(period='P2', term=term),
        delta_term.sel(term=term),
    ]

    row_max = max(float(np.abs(arr).max().compute()) for arr in row_arrays)
    if np.isclose(row_max, 0.0):
        row_max = 1.0e-12
    levels = np.linspace(-row_max, row_max, 17)
    ticks = np.linspace(-row_max, row_max, 5)

    mesh = None
    for idx, (ax, arr, (panel_key, panel_title, _, _)) in enumerate(zip(axes, row_arrays, panel_defs)):
        data = arr.compute()
        mesh = data.plot.pcolormesh(
            ax=ax,
            transform=ccrs.PlateCarree(),
            cmap=map_cmap,
            levels=levels,
            extend='both',
            add_colorbar=False,
            infer_intervals=True,
        )

        ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
        ax.set_extent(MC_EXTENT, crs=ccrs.PlateCarree())
        ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
        ax.set_xticks(MC_XTICKS, crs=ccrs.PlateCarree())
        ax.set_yticks(MC_YTICKS, crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(direction='out', top=True, right=True, labelsize=9)
        ax.set_title(panel_title, fontsize=12, fontweight='bold')
        add_target_box(ax)

        if idx > 0:
            ax.set_yticklabels([])
        if idx < 2:
            ax.set_xticklabels([])

    axes[0].set_ylabel(term_labels[term], fontsize=12, fontweight='bold')
    fig.suptitle(f'Grid-wise ENSO-regressed lower-tropospheric MFC decomposition: {term_labels[term]}', fontsize=15, fontweight='bold')

    cbar = fig.colorbar(mesh, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend='both')
    cbar.set_label('kg m$^{-2}$ s$^{-1}$ per 1 std Nino3.4', fontsize=11)
    cbar.ax.tick_params(labelsize=9)

    fig.tight_layout(rect=[0, 0.02, 1, 0.95])
    fig.savefig(NOTEBOOK_DIR / f'moisture_decomp_gridwise_TARGETBOX_{term}.png', dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Saved figure -> {NOTEBOOK_DIR / f"moisture_decomp_gridwise_TARGETBOX_{term}.png"}')

# The saved NetCDF can be used to revise plots without recomputing the decomposition.
